# Trend-Filtered Peg Arbitrage: USDe (Ethena) — a Structurally Different Case

Fourth pair in this series, and a deliberately different kind of test. The first three notebooks
(PAXG/gold-perp, stETH/ETH, WBTC/BTC) all compare **two instruments tracking the same volatile
underlying** — trend on the underlying is a natural predictor to test against the basis between them.
**USDe** doesn't fit that shape: it's Ethena Labs' synthetic dollar, engineered to stay near **$1**,
not to track a trending asset. So the trend signal here can't be USDe's own price — that would just
be testing "does a stablecoin trend," which it isn't supposed to do.

**What actually varies with trend here**: USDe's peg is backed by a delta-neutral strategy — hold
staked ETH (or other liquid-staking collateral), short an equivalent notional of ETH perpetual
futures. Its yield comes from *staking rewards plus perpetual funding payments*, and funding tends to
run strongly **positive** when the market is broadly long/bullish (leveraged longs pay shorts) and
can compress or go negative during sharp downtrends. That gives a real, mechanistic reason to test
**ETH's trend** — not USDe's own price — as a predictor of USDe's peg deviation: strong ETH uptrends
should coincide with richer funding income (healthier backing, more confidence, more inflows chasing
yield → tighter peg or slight premium); sharp downtrends should coincide with strained funding income
and redemption pressure → wider discount.

**Hypothesis, stated directionally**: `eth_trend_pct` and `usde_basis_z` should be **positively
correlated** — the same directional logic as the other three notebooks, but through a genuinely
different causal channel (yield/confidence in a synthetic-dollar mechanism, not direct wrap/collateral
basis on the trending asset itself).

**Data source**: Kraken's public API (free, no key) for both `USDEUSD` and `ETHUSD`. USDe is a newer
product — its Kraken listing only goes back to ~September 2025, giving roughly 11 months of history,
the shortest sample in this series.

> ⚠️ Research only, not investment advice. Models a market-neutral spread position (long USDe / short
> ETH exposure sized to the basis, or the reverse) with no transaction costs, redemption mechanics, or
> execution slippage. This is the most speculative notebook in the series — treat findings as a
> starting hypothesis to refine (e.g. with actual ETH perpetual funding-rate data instead of price
> trend as the predictor) rather than a tested result.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install pandas numpy matplotlib seaborn scipy statsmodels requests

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.5f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data Collection


In [ ]:
KRAKEN_BASE = "https://api.kraken.com/0/public/OHLC"


def fetch_kraken_ohlc(pair, interval=1440):
    r = requests.get(KRAKEN_BASE, params={"pair": pair, "interval": interval}, timeout=20)
    r.raise_for_status()
    payload = r.json()
    if payload.get("error"):
        raise RuntimeError(payload["error"])
    result_key = [k for k in payload["result"] if k != "last"][0]
    rows = payload["result"][result_key]
    df = pd.DataFrame(rows, columns=["ts", "open", "high", "low", "close", "vwap", "volume", "count"])
    df["date"] = pd.to_datetime(df["ts"].astype(np.int64), unit="s").dt.normalize()
    df = df.drop_duplicates(subset="date").set_index("date").sort_index()
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = df[c].astype(float)
    return df[["open", "high", "low", "close", "volume"]]


usde_usd = fetch_kraken_ohlc("USDEUSD")
eth_usd = fetch_kraken_ohlc("ETHUSD")

print(f"USDEUSD: {usde_usd.index.min().date()} to {usde_usd.index.max().date()}, {len(usde_usd)} days")
print(f"ETHUSD:  {eth_usd.index.min().date()} to {eth_usd.index.max().date()}, {len(eth_usd)} days")


## 2. Basis Construction (Peg Deviation)


In [ ]:
df = pd.DataFrame({
    "usde_usd": usde_usd["close"],
    "eth_usd": eth_usd["close"],
}).dropna()

df["basis_pct"] = df["usde_usd"] - 1  # deviation from the $1 peg

print(f"Overlapping sample: {df.index.min().date()} to {df.index.max().date()}, {len(df)} days")
df.tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df.index, df["eth_usd"], color="tab:blue")
axes[0].set_title("ETH-USD Price")

axes[1].plot(df.index, df["basis_pct"] * 100, color="tab:green")
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("USDe Peg Deviation: (USDe/USD - 1) x 100%")
axes[1].set_ylabel("%")
plt.tight_layout()
plt.show()

print(f"Basis stats: mean={df['basis_pct'].mean()*100:.4f}%, std={df['basis_pct'].std()*100:.4f}%, "
      f"min={df['basis_pct'].min()*100:.4f}%, max={df['basis_pct'].max()*100:.4f}%")


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_stat, p_value, *_ = adfuller(df["basis_pct"].dropna())
print(f"ADF test on basis_pct: statistic={adf_stat:.3f}, p-value={p_value:.4f}")
print("p < 0.05 suggests the basis is stationary (mean-reverting) over this sample.")


## 3. Trend Detection — on ETH, the Proxy Variable (Not USDe Itself)


In [ ]:
def find_pivots(price, window=5):
    is_high = price == price.rolling(window * 2 + 1, center=True).max()
    is_low = price == price.rolling(window * 2 + 1, center=True).min()
    return price[is_high], price[is_low]


def fit_trendline(pivot_points, n_recent=3):
    if len(pivot_points) < 2:
        return None
    recent = pivot_points.iloc[-n_recent:]
    x = (recent.index - recent.index[0]).days.values.astype(float)
    y = recent.values
    slope, intercept = np.polyfit(x, y, 1)
    return recent.index[0], slope, intercept


swing_highs, swing_lows = find_pivots(df["eth_usd"], window=5)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df.index, df["eth_usd"], color="tab:blue", label="ETH-USD close", zorder=1)
ax.scatter(swing_highs.index, swing_highs.values, color="red", s=25, zorder=3, label="swing highs")
ax.scatter(swing_lows.index, swing_lows.values, color="green", s=25, zorder=3, label="swing lows")

for pivots, color in [(swing_highs, "red"), (swing_lows, "green")]:
    fit = fit_trendline(pivots, n_recent=4)
    if fit:
        x0, slope, intercept = fit
        x_line = pd.date_range(x0, df.index[-1])
        y_line = intercept + slope * (x_line - x0).days
        ax.plot(x_line, y_line, color=color, linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_title("ETH Price with Recent Swing-High / Swing-Low Trend Lines")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
def rolling_trend_pct(price, window=20):
    def slope(y):
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]
    raw_slope = price.rolling(window).apply(slope, raw=True)
    return raw_slope * window / price

df["trend_pct"] = rolling_trend_pct(df["eth_usd"], window=20)

fig, ax = plt.subplots()
df["trend_pct"].mul(100).plot(ax=ax, color="purple")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("Rolling 20-Day ETH Trend Signal (Implied %% Move Over Window)")
ax.set_ylabel("%")
plt.tight_layout()
plt.show()


## 4. Does ETH's Trend Predict USDe's Peg Deviation?


In [ ]:
def zscore(s, window=20):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

df["basis_z"] = zscore(df["basis_pct"], window=20)

sub = df[["trend_pct", "basis_z"]].dropna()
corr_val = sub["trend_pct"].corr(sub["basis_z"])

fig, ax = plt.subplots()
ax.scatter(sub["trend_pct"] * 100, sub["basis_z"], alpha=0.5, s=20)
ax.set_xlabel("Rolling ETH trend signal (%% move over 20d)")
ax.set_ylabel("USDe peg-deviation z-score")
ax.set_title(f"ETH Trend vs. USDe Peg Deviation (n={len(sub)}, corr={corr_val:.3f})")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
plt.tight_layout()
plt.show()

print(f"Correlation(trend_pct, basis_z) = {corr_val:.3f}")
if abs(corr_val) < 0.15:
    print("Near zero: no meaningful relationship in this sample — the funding/confidence hypothesis "
          "isn't showing up via price trend. Direct ETH perpetual funding-rate data (not price trend) "
          "may be a more mechanistically direct predictor worth testing instead.")
elif corr_val > 0:
    print("Positive, as hypothesized: ETH uptrends coincide with USDe trading richer/tighter, "
          "downtrends with a wider discount.")
else:
    print("Negative — the opposite of the hypothesis. Worth investigating why before leaning on it.")


## 5. Mean-Reversion Arbitrage Signal

**Position convention**: `+1` = long the basis (long USDe / short an equivalent $ notional elsewhere,
taken when USDe is *below* its own recent-average peg, betting it reverts up). `-1` = short the basis
(taken when USDe is trading rich relative to its own recent history).


In [ ]:
def build_positions(z, entry=2.0, exit=0.5, blocked=None):
    position = pd.Series(0.0, index=z.index)
    current = 0.0
    for i, (idx, val) in enumerate(z.items()):
        if pd.isna(val):
            position.iloc[i] = current
            continue
        is_blocked = bool(blocked.loc[idx]) if blocked is not None and idx in blocked.index else False
        if current == 0.0:
            if not is_blocked:
                if val > entry:
                    current = -1.0
                elif val < -entry:
                    current = 1.0
        else:
            if abs(val) < exit:
                current = 0.0
        position.iloc[i] = current
    return position


ENTRY_Z, EXIT_Z = 2.0, 0.5
position_pure_mr = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
df["basis_z"].plot(ax=axes[0], color="tab:green")
axes[0].axhline(ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(-ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(0, color="gray", linewidth=0.8)
axes[0].set_title("USDe Peg-Deviation Z-Score with Entry Thresholds")

position_pure_mr.plot(ax=axes[1], color="black", drawstyle="steps-post")
axes[1].set_title("Pure Mean-Reversion Position (+1 = long basis, -1 = short basis)")
plt.tight_layout()
plt.show()


## 6. Combining Both Ideas: Trend-Filtered Arbitrage


In [ ]:
trend_strength = df["trend_pct"].abs()
trend_threshold = trend_strength.rolling(90, min_periods=30).quantile(0.67)
strong_trend = trend_strength > trend_threshold

position_trend_filtered = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z, blocked=strong_trend)

print(f"Days blocked by trend filter: {strong_trend.sum()} / {strong_trend.notna().sum()} "
      f"({strong_trend.mean()*100:.1f}%)")


In [ ]:
def sharpe(returns, periods_per_year=365):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)


def max_drawdown(cum_returns):
    running_max = cum_returns.cummax()
    dd = cum_returns / running_max - 1
    return dd.min()


basis_chg = df["basis_pct"].diff()

strategies = pd.DataFrame({
    "pure_mean_reversion": position_pure_mr.shift(1) * basis_chg,
    "trend_filtered_mean_reversion": position_trend_filtered.shift(1) * basis_chg,
})

cum = (1 + strategies.fillna(0)).cumprod()
cum.plot(title="Cumulative Return: Pure vs. Trend-Filtered USDe Peg Arbitrage")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "annualized_sharpe": strategies.apply(sharpe),
    "total_return": cum.iloc[-1] - 1,
    "annualized_vol": strategies.std() * np.sqrt(365),
    "max_drawdown": cum.apply(max_drawdown),
    "n_trades": pd.Series({
        "pure_mean_reversion": (position_pure_mr.diff().abs() > 0).sum(),
        "trend_filtered_mean_reversion": (position_trend_filtered.diff().abs() > 0).sum(),
    }),
})
summary


## 7. Market-Neutrality Check

For USDe this check has a different meaning than the other three notebooks — a well-designed peg
strategy should be uncorrelated with ETH's return *directly*, since ETH exposure is supposed to be
fully hedged out by construction. A strong correlation here would be a red flag that this strategy is
accidentally carrying delta-one crypto exposure it isn't supposed to have.


In [ ]:
eth_ret = np.log(df["eth_usd"]).diff()

exposure_check = pd.DataFrame({
    "pure_mr_vs_eth": [strategies["pure_mean_reversion"].corr(eth_ret)],
    "trend_filtered_vs_eth": [strategies["trend_filtered_mean_reversion"].corr(eth_ret)],
}, index=["correlation_with_eth_return"])
exposure_check


## 8. Does the Gross Edge Survive Real Trading Costs?


In [ ]:
TAKER_FEE_PER_FILL = 0.0005
FILLS_PER_ROUND_TRIP = 4

fee_drag = summary["n_trades"] * FILLS_PER_ROUND_TRIP * TAKER_FEE_PER_FILL
net_return_estimate = summary["total_return"] - fee_drag

cost_comparison = pd.DataFrame({
    "gross_total_return": summary["total_return"],
    "est_fee_drag": fee_drag,
    "est_net_total_return": net_return_estimate,
})
cost_comparison


## 9. Takeaways & Next Steps

- **This is the weakest-grounded hypothesis of the four, by design** — it substitutes a proxy
  (ETH price trend) for the thing that's actually mechanistically closest to USDe's peg health
  (ETH perpetual funding rates). If Section 4 came back weak, the natural next step isn't to discard
  the idea, it's to **rebuild `trend_pct` from actual funding-rate data** (Binance/OKX/Bybit perpetual
  funding history) instead of ETH price momentum — a more direct test of the real mechanism.
- **Shortest sample in this series** (~11 months, Kraken's USDe listing start) — treat any result here
  as the most provisional of the four notebooks.
- **The market-neutrality check means something different here**: a "clean" arb strategy on a wrap/
  stake pair (the other three notebooks) is expected to be uncorrelated with the underlying almost
  automatically, by construction of a long-one/short-the-other trade. For USDe, this check is really
  asking whether the *strategy itself* (Ethena's, not just this backtest's) is doing its job — a
  meaningfully non-zero correlation would be a legitimate red flag about the peg mechanism's health,
  not just a backtest artifact.
- **Real-world execution isn't modeled**: USDe redemption/minting mechanics, Ethena's own cooldown
  periods, and counterparty risk on the perpetual short leg are all real, material frictions absent
  from this backtest.
- **Robustness**: validate out-of-sample once more history accumulates, rebuild with funding-rate
  data as suggested above, and account for transaction costs (Section 8) before treating any of this
  as an actual edge.
